# CPU vs GPU Intuition Lab

Trainer voice: we will compare the same operation on CPU and GPU in small chunks.

**Outcome:** explain throughput differences with evidence.

## Step 1 - Prediction first

Pause and ask: **Which device should finish faster for large matrix multiply?**

In [ ]:
import time
import torch

In [ ]:
print("CUDA available:", torch.cuda.is_available())

## Step 2 - CPU baseline

Expected: this should run, but usually slower for this workload size.

In [ ]:
x_cpu = torch.randn((1024, 1024))
y_cpu = torch.randn((1024, 1024))

In [ ]:
t0 = time.perf_counter()
_ = torch.matmul(x_cpu, y_cpu)
cpu_time = time.perf_counter() - t0
print(round(cpu_time, 4))

## Step 3 - GPU run

Common mistake: forgetting synchronization around timing.

In [ ]:
if torch.cuda.is_available():
    x_gpu = x_cpu.to("cuda")
    y_gpu = y_cpu.to("cuda")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.synchronize()
    t1 = time.perf_counter()

In [ ]:
if torch.cuda.is_available():
    _ = torch.matmul(x_gpu, y_gpu)
    torch.cuda.synchronize()
    print(round(time.perf_counter() - t1, 4))

## Checkpoint — Why was the GPU faster (or not)?

**The questions we asked:**
- What made GPU faster or not faster?
- How would workload size change this result?

**Model answer (strong understanding)**

For a large matrix multiplication (1024×1024 or bigger), the GPU is dramatically faster because the operation is *embarrassingly parallel* and *compute-bound*. Every element of the output matrix can be computed independently, and there are millions of such elements. A modern GPU has thousands of simple cores that can all be doing fused multiply-adds at the same time. The CPU, even with good vectorization (AVX-512 etc.), has far fewer powerful cores and must serialize or coarsely parallelize the work.

The key limiter is not raw clock speed — it is *parallel throughput* and *memory bandwidth* once the data is on the device. In this lab the tensors were already on the target device, so we mostly measured arithmetic throughput. That is why you typically see 10-100× speedups on this exact micro-benchmark.

Workload size matters enormously. For tiny matrices (say 32×32) the GPU can actually be *slower* once you include kernel launch overhead and the cost of getting data to the device. The crossover point is workload-dependent; this is why real engineers always measure, never assume "bigger is always better on GPU."

**If your intuition went the other way, here is what usually confuses people:**
They picture the CPU as "general purpose and therefore faster at everything" or they think about single-threaded latency instead of massive parallel throughput. The mental shift is: GPUs are not faster CPUs; they are *throughput monsters for the right shape of problem*.

**Common misconception**

"If I just buy a more expensive CPU with more cores I can match the GPU."

This is rarely true for the workloads that matter in modern AI and data systems. Even a 128-core CPU cannot come close to a mid-range GPU for dense linear algebra at scale, because the GPU's architecture, memory hierarchy, and programming model are specialized for exactly this pattern. The economic reality in 2026 is that for training or large-batch inference, you buy GPUs, not 256-core CPUs.

**If the speedup you saw was smaller or larger than expected**

Welcome to real life. Different Colab instances give different GPUs (T4, V100, A100, etc.). Different matrix sizes hit different bottlenecks (compute vs memory bandwidth). If you saw only 2-3×, you may have been measuring something closer to the memory-bound regime, or the CPU was unusually well vectorized for that Colab CPU. The number is less important than the *direction* and the *habit of asking why*.


## Lesson Recap — What You Actually Learned

- You have direct experiential evidence that the same mathematical operation has radically different wall-clock cost depending on which processor you give it.
- You saw (and can now explain) that parallelism + memory hierarchy, not clock speed, is usually the deciding factor for GPU wins.
- You practiced the discipline of keeping tensors on the device and using synchronization so your measurements are honest.
- You discovered that "GPU is faster" is not a universal law — it is a statement about workload shape, data location, and measurement honesty.

**Human note:** If part of you still feels like the GPU result was somehow "cheating" or magic, that reaction is normal. You just watched thousands of simple cores do in 10 ms what a handful of very smart cores needed hundreds of milliseconds to do. The rest of the course is about turning that "wow" into deliberate engineering decisions.


## Role Lens — Why This Matters in Real Work

**DevOps / MLOps** — When a training job is using 8 GPUs but only showing 30% utilization, the first question is often "is this workload even the right shape for a GPU?" The intuition you built here ("is the operation massively parallel and compute-heavy enough?") is exactly how you decide whether to fight for more GPU quota or to tell the team to profile the data movement first.

**Data Science** — Every time you choose to move a model from CPU prototyping to GPU training, you are making a bet that the extra complexity (device placement, synchronization, memory management) will be paid back in speed. The micro-benchmark habit you just practiced is how you make that bet with data instead of hope.

**Data Engineering** — Many "GPU for data" projects die in the first week because the team moved a pandas transform to cuDF expecting 50× and got 0.8× once the copy cost was included. The same "does the workload shape actually favor the GPU?" question applies to ETL just as much as to training.

---

**You now have the first piece of real GPU intuition.**

Next section we go deeper into the "move once, compute many times" rule and start writing code that actually respects the GPU's reality instead of fighting it. You are ready.
